# 09 · Baselines for trajectory and risk (P3)

The simple methods every model must beat. Fitted on training grafts; scored on validation grafts and by 5×5 repeated cross-validation; test grafts untouched. Code: `trajectory/baselines.py`, `risk/cox.py`, `evaluation/followup_eval.py`. Reads `outputs/followup_baselines/baselines.json` (run `python -m recommendation.main followup-baselines` to refresh).

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
r = json.loads((OUTPUTS / "followup_baselines" / "baselines.json").read_text())
print("Data source:", r["source"], "| train", r["train_grafts"], "| CV grafts", r["cv_grafts"])

## 1. Trajectory: error of predicted ECD (MAE, cells/mm²; repeated CV)

In [ ]:
pd.DataFrame({n: {h: d["repeated_cv"][h]["mae"] for h in ("24m", "36m", "60m", "all")}
              for n, d in r["trajectory"].items()}).T.round(0)

## 2. Risk (C-index, time-dependent AUC, Brier; repeated CV)

In [ ]:
for endpoint, models in r["risk"].items():
    print(endpoint)
    display(pd.DataFrame({n: d["repeated_cv"] for n, d in models.items()}).T.round(3))